# 04 - ML Classification
**FINAL: TF-IDF/BoW, Train/Test Split, Naive Bayes, Evaluation**

In [ ]:
# === IMPORTS ===
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

---
## PATTERN 1: TF-IDF Vectorization

In [ ]:
# Sample texts
texts = [
    "I love this movie, it was great!",
    "This film was terrible and boring.",
    "Amazing performance by the actors.",
    "Worst movie I have ever seen."
]

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(lowercase=True, stop_words='english')
X = tfidf.fit_transform(texts)

print('Shape:', X.shape)
print('Features:', tfidf.get_feature_names_out())

In [ ]:
# Display as dense array
print('TF-IDF Matrix:')
print(X.toarray())

---
## PATTERN 2: Bag of Words (CountVectorizer)

In [ ]:
# Count Vectorizer (Bag of Words)
count_vec = CountVectorizer(lowercase=True, stop_words='english')
X_bow = count_vec.fit_transform(texts)

print('Shape:', X_bow.shape)
print('Features:', count_vec.get_feature_names_out())
print('BoW Matrix:')
print(X_bow.toarray())

---
## PATTERN 3: Train/Test Split

In [ ]:
# Sample data
texts = ["great movie", "terrible film", "loved it", "hated it", 
         "amazing", "awful", "fantastic", "boring"]
labels = ['pos', 'neg', 'pos', 'neg', 'pos', 'neg', 'pos', 'neg']

# 70/30 split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, 
    test_size=0.3,       # 30% for test
    random_state=42,     # reproducibility
    stratify=labels      # maintain class balance
)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')

---
## PATTERN 4: Naive Bayes Classification

In [ ]:
# Step 1: Vectorize
vectorizer = TfidfVectorizer(lowercase=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)  # Use transform, not fit_transform!

# Step 2: Train Naive Bayes
clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

# Step 3: Predict
predictions = clf.predict(X_test_vec)
print('Predictions:', predictions)

---
## PATTERN 5: Model Evaluation

In [ ]:
# Accuracy
acc = accuracy_score(y_test, predictions)
print(f'Accuracy: {acc:.4f} ({acc*100:.2f}%)')

In [ ]:
# Classification Report (precision, recall, f1)
print('Classification Report:')
print(classification_report(y_test, predictions))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, predictions, labels=['pos', 'neg'])
print('Confusion Matrix:')
print(cm)

# Visual confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['pos', 'neg'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

---
## PATTERN 6: Using Pipeline (Recommended)

In [ ]:
# Pipeline combines vectorization + classification
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english')),
    ('clf', MultinomialNB())
])

# Train (no need to vectorize separately)
pipeline.fit(X_train, y_train)

# Predict
predictions = pipeline.predict(X_test)

# Evaluate
print(f'Accuracy: {accuracy_score(y_test, predictions):.4f}')

---
## COMPLETE EXAMPLE: Sentiment Classification

In [ ]:
# Load movie reviews data (from NLTK)
import nltk
from nltk.corpus import movie_reviews
import random

# Build dataset
docs = [(movie_reviews.raw(fid), movie_reviews.categories(fid)[0])
        for fid in movie_reviews.fileids()]

random.seed(42)
random.shuffle(docs)

texts, labels = zip(*docs)
print(f'Total documents: {len(texts)}')
print(f'Labels: {set(labels)}')

In [ ]:
# Train/Test Split (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# Build and train pipeline
model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        max_df=0.95,    # Ignore very common words
        min_df=2        # Ignore very rare words
    )),
    ('clf', MultinomialNB())
])

model.fit(X_train, y_train)

In [ ]:
# Evaluate
predictions = model.predict(X_test)

print('=== Model Evaluation ===')
print(f'Accuracy: {accuracy_score(y_test, predictions):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, predictions))

In [ ]:
# Predict on new text
new_review = "This movie was absolutely fantastic! I loved every minute."
prediction = model.predict([new_review])
print(f'New review prediction: {prediction[0]}')

---
## ALTERNATIVE: Logistic Regression (from lectures)

In [ ]:
# Logistic Regression pipeline (used in Lab 6)
model_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(1, 2),  # Include bigrams
        max_df=0.95,
        min_df=2
    )),
    ('clf', LogisticRegression(max_iter=200))
])

model_lr.fit(X_train, y_train)
predictions_lr = model_lr.predict(X_test)
print(f'Logistic Regression Accuracy: {accuracy_score(y_test, predictions_lr):.4f}')

---
## QUICK REFERENCE: Key Parameters

In [ ]:
# TfidfVectorizer parameters:
# - lowercase=True: convert to lowercase
# - stop_words='english': remove common English words
# - max_df=0.95: ignore words in >95% of docs
# - min_df=2: ignore words in <2 docs
# - ngram_range=(1,2): use unigrams and bigrams

# train_test_split parameters:
# - test_size=0.3: 30% for test
# - random_state=42: reproducibility
# - stratify=labels: maintain class proportions

# MultinomialNB: No required parameters

# LogisticRegression parameters:
# - max_iter=200: max iterations for convergence
# - C=1.0: regularization strength

print('Parameters reference loaded!')